In [1]:
from __future__ import annotations

import json
import random
import sys
import time
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from tqdm import tqdm
from xgboost import XGBClassifier

ROOT = Path().resolve()
base_path = '/Data/yuhan.wu/kaggle_project_new'
sys.path.append(f'{base_path}/src')

from churn_pipeline.dataset_builder import build_datasets_cli  # noqa: E402
from churn_pipeline.transformer_user_day import (  # noqa: E402
    ChurnTransformerUserDay,
    TrainingConfig,
    UserSequenceDataset,
    build_user_day_sequences,
    collate_batch,
    make_dataloaders,
    predict_proba,
    train_model,
)

## 1) Configuration: cutoffs, label, feature window, model params, blend params

In [ ]:
# Data paths
COMPETITION_DIR = Path(f"{base_path}/churn-prediction-25-26")
TRAIN_PATH = COMPETITION_DIR / 'train.parquet'
TEST_PATH = COMPETITION_DIR / 'test.parquet'
EXAMPLE_SUB = COMPETITION_DIR / 'example_submission.csv'

# rolling cutoffs（先稀疏跑通，再加密）
CUTOFF_START = '2018-10-20'
CUTOFF_END = '2018-11-10'
CUTOFF_FREQ = '2D'  # '1D' | '2D' | '3D' | ...
CUT_OFFS = pd.date_range(CUTOFF_START, CUTOFF_END, freq=CUTOFF_FREQ)

# 标签
LABEL_MODE = 'horizon'  # 'horizon' | 'ever'
HORIZON_DAYS = 10

# 特征窗口：看最近多少天（与 Transformer max_seq_len 对齐）
LOOKBACK_DAYS = 51

# 防泄露过滤（与主 notebook 对齐）
EXCLUDE_PAGES = 'Cancellation Confirmation,Cancel'
EXCLUDE_AUTH_VALUES = 'Cancelled'

# build 参数
DROP_LAST_K_EVENTS = 0
FULL_CALENDAR = True
VAL_PROB_LOW = 0.8
VAL_PROB_HIGH = 1.0
SPLIT_SEED = 'split_v1'
CACHE_DIR = '.cache/user_day_features'
OUTPUT_BASE_DIR = 'data/processed'
BATCH_SIZE = 1_000_000
FORCE_RECOMPUTE = False

# Transformer 训练配置（这里独立调参）
t_config = TrainingConfig(
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.5,
    max_seq_len=int(LOOKBACK_DAYS),
    batch_size=256,
    lr=5e-4,
    weight_decay=5e-3,
    epochs=25,
    use_cosine_decay=True,
    eta_min_factor=0.1,
    warmup_epochs=2,
    use_focal_loss=True,
    focal_gamma=2.0,
    early_stop_patience=3,
    early_stop_min_delta=0.0,
    pooling='mean'
)
T_SEEDS = [42, 2021, 7]
T_POS_WEIGHT = 7.0

# XGB 参数（这里独立调参）
XGB_DEVICE = 'cuda'  # 'cuda' | 'cpu'
xgb_params = dict(
    n_estimators=5000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=2.0,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=0.5,
    reg_alpha=0.5,
    gamma=0.0,
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    device=XGB_DEVICE,
    random_state=42,
    n_jobs=-1,
)
XGB_SCALE_POS_WEIGHT = 5.0
EARLY_STOPPING_ROUNDS = 200
XGB_CALLBACKS = [
    xgb.callback.EarlyStopping(
        rounds=int(EARLY_STOPPING_ROUNDS),
        metric_name='auc',
        data_name='validation_0',
        maximize=True,
        save_best=False,
    )
]

# 融合（logit blend）参数
BLEND_W_GRID_COARSE = 21  # w in [0,1]
BLEND_W_FINE_SPAN = 0.10
BLEND_W_GRID_FINE = 41

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('cutoffs=', len(CUT_OFFS), CUT_OFFS.min().date(), '->', CUT_OFFS.max().date())
print('device=', device)
print('max_seq_len=', t_config.max_seq_len)

cutoffs= 11 2018-10-20 -> 2018-11-09
device= cuda
max_seq_len= 51


## 2) Build rolling train/val (concatenate multiple cutoff snapshots)

Outputs:
- `train_ud_all` / `val_ud_all`: concatenated user-day table (userId has a `|cutoff` suffix to avoid collisions)
- `labels_all`: train+val labels (userId also has the suffix)


In [3]:
t0 = time.time()
all_train_ud = []
all_val_ud = []
all_labels = []

for cutoff in tqdm(CUT_OFFS, desc='build rolling cutoffs'):
    cutoff_str = str(pd.Timestamp(cutoff).date())
    paths = build_datasets_cli(
        train_path=str(TRAIN_PATH),
        test_path=str(TEST_PATH),
        output_dir=None,
        output_base_dir=str(OUTPUT_BASE_DIR),
        cache_dir=str(CACHE_DIR),
        cutoff_time=cutoff_str,
        horizon_days=int(HORIZON_DAYS),
        label_mode=str(LABEL_MODE),
        drop_last_k_events=int(DROP_LAST_K_EVENTS),
        exclude_pages=str(EXCLUDE_PAGES),
        exclude_auth_values=str(EXCLUDE_AUTH_VALUES),
        val_prob_low=float(VAL_PROB_LOW),
        val_prob_high=float(VAL_PROB_HIGH),
        split_seed=str(SPLIT_SEED),
        batch_size=int(BATCH_SIZE),
        full_calendar=bool(FULL_CALENDAR),
        force_recompute=bool(FORCE_RECOMPUTE),
        verbose=False,
    )

    train_ud = pd.read_parquet(paths['train_features'])
    val_ud = pd.read_parquet(paths['val_features'])
    train_lab = pd.read_parquet(paths['train_labels'])[['userId', 'label']]
    val_lab = pd.read_parquet(paths['val_labels'])[['userId', 'label']]

    end_day = pd.Timestamp(cutoff).normalize() - pd.Timedelta(days=1)
    start_day = end_day - pd.Timedelta(days=int(LOOKBACK_DAYS) - 1)

    def _slice(df: pd.DataFrame) -> pd.DataFrame:
        d = pd.to_datetime(df['day']).dt.normalize()
        return df.loc[(d >= start_day) & (d <= end_day)].copy()

    train_ud = _slice(train_ud)
    val_ud = _slice(val_ud)

    suffix = '|' + cutoff_str
    train_ud['userId'] = train_ud['userId'].astype(str) + suffix
    val_ud['userId'] = val_ud['userId'].astype(str) + suffix
    train_lab['userId'] = train_lab['userId'].astype(str) + suffix
    val_lab['userId'] = val_lab['userId'].astype(str) + suffix

    all_train_ud.append(train_ud)
    all_val_ud.append(val_ud)
    all_labels.append(pd.concat([train_lab, val_lab], axis=0, ignore_index=True))

train_ud_all = pd.concat(all_train_ud, axis=0, ignore_index=True)
val_ud_all = pd.concat(all_val_ud, axis=0, ignore_index=True)
labels_all = pd.concat(all_labels, axis=0, ignore_index=True)
labels_dict = {str(uid): int(y) for uid, y in zip(labels_all['userId'].astype(str), labels_all['label'].astype(int))}

print('train_ud_all', train_ud_all.shape, 'val_ud_all', val_ud_all.shape)
print('labels_all', labels_all.shape, 'unique label keys', len(labels_dict))
print('seconds', round(time.time() - t0, 2))
(train_ud_all.head(), val_ud_all.head(), labels_all.head())

build rolling cutoffs: 100%|██████████| 11/11 [08:42<00:00, 47.53s/it]


train_ud_all (4872087, 105) val_ud_all (1233573, 105)
labels_all (210540, 2) unique label keys 210540
seconds 523.26


(               userId        day  d_events  d_sessions  d_pages_unique  \
 0  1000025|2018-10-20 2018-10-01         0           0               0   
 1  1000025|2018-10-20 2018-10-02       120           2              12   
 2  1000025|2018-10-20 2018-10-03       381           0              11   
 3  1000025|2018-10-20 2018-10-04       237           1               8   
 4  1000025|2018-10-20 2018-10-05        27           1               4   
 
    d_active_hours  d_first_hour  d_last_hour  d_nextsong_cnt  \
 0               0            -1           -1               0   
 1               9             8           23              95   
 2              22             0           21             317   
 3              14             1           14             201   
 4               3             1            3              22   
 
    d_listen_sec_sum  ...  x_active_streak  d_events_per_session  \
 0          0.000000  ...                0                   0.0   
 1      24922.318359

## 3) Build test with a fixed cutoff (for submission)

Note: test is not built in a rolling way; we use a single cutoff (usually the last day).


In [4]:
SUBMISSION_CUTOFF = str(pd.Timestamp(CUTOFF_END).date())
sub_paths = build_datasets_cli(
    train_path=str(TRAIN_PATH),
    test_path=str(TEST_PATH),
    output_dir=None,
    output_base_dir=str(OUTPUT_BASE_DIR),
    cache_dir=str(CACHE_DIR),
    cutoff_time=SUBMISSION_CUTOFF,
    horizon_days=int(HORIZON_DAYS),
    label_mode=str(LABEL_MODE),
    drop_last_k_events=int(DROP_LAST_K_EVENTS),
    exclude_pages=str(EXCLUDE_PAGES),
    exclude_auth_values=str(EXCLUDE_AUTH_VALUES),
    val_prob_low=float(VAL_PROB_LOW),
    val_prob_high=float(VAL_PROB_HIGH),
    split_seed=str(SPLIT_SEED),
    batch_size=int(BATCH_SIZE),
    full_calendar=bool(FULL_CALENDAR),
    force_recompute=bool(FORCE_RECOMPUTE),
    verbose=False,
)

test_ud = pd.read_parquet(sub_paths['test_features'])
cutoff_ts = pd.Timestamp(SUBMISSION_CUTOFF).normalize()
end_day = cutoff_ts - pd.Timedelta(days=1)
start_day = end_day - pd.Timedelta(days=int(LOOKBACK_DAYS) - 1)
d = pd.to_datetime(test_ud['day']).dt.normalize()
test_ud = test_ud.loc[(d >= start_day) & (d <= end_day)].copy()

print('test_ud', test_ud.shape, 'window', str(start_day.date()), '->', str(end_day.date()))
test_ud.head()

test_ud (116160, 105) window 2018-09-20 -> 2018-11-09


,userId,day,d_events,d_sessions,d_pages_unique,d_active_hours,d_first_hour,d_last_hour,d_nextsong_cnt,d_listen_sec_sum,...,x_active_streak,d_events_per_session,d_nextsong_per_session,x_roll7_std_events,x_roll7_std_listen_sec_mean,d_day_of_week,x_today_over_roll7_events_mean,x_today_over_roll7_nextsong_mean,x_roll7_minus_roll28_events_mean,x_roll7_minus_roll28_nextsong_mean
0,1000655,2018-10-01,0,0,0,0,-1,-1,0,0.000000,...,0,0.0,0.0,0.000000,0.000000,0,0.0,0.000000,0.000000,0.000000
1,1000655,2018-10-02,0,0,0,0,-1,-1,0,0.000000,...,0,0.0,0.0,0.000000,0.000000,1,0.0,0.000000,0.000000,0.000000
2,1000655,2018-10-03,39,1,6,3,9,11,31,8547.356445,...,1,39.0,31.0,13.647158,96.482307,2,7.0,7.000000,4.178572,3.321428
3,1000655,2018-10-04,0,0,0,0,-1,-1,0,0.000000,...,0,0.0,0.0,13.647158,96.482307,3,0.0,0.000000,4.178572,3.321428
4,1000655,2018-10-05,31,1,7,2,22,23,23,7123.059082,...,1,31.0,23.0,15.955296,132.544189,4,3.1,2.981481,7.500000,5.785714


## 4) Transformer feature processing: drop constant cols + log1p/clip + StandardScaler

Keep this consistent with the single-model Transformer notebook:
- Feature columns: all columns except `userId`, `day`
- Drop columns that are constant in rolling train
- Apply log1p/clip to count-like features
- Fit scaler on rolling train only; apply to val/test


In [5]:
train_ud_t = train_ud_all.copy()
val_ud_t = val_ud_all.copy()
test_ud_t = test_ud.copy()

feature_cols = [c for c in train_ud_t.columns if c not in {'userId', 'day'}]
feature_cols = sorted(feature_cols)

_nuniq = train_ud_t[feature_cols].nunique(dropna=False)
keep_cols = [c for c in feature_cols if int(_nuniq[c]) > 1]
drop_cols = [c for c in feature_cols if c not in set(keep_cols)]
print('drop constant cols:', len(drop_cols))
feature_cols = keep_cols

def _is_count_like(col: str) -> bool:
    if col.endswith('_cnt') or col.endswith('_sum'):
        return True
    if col.endswith('_active_days') or col.endswith('_paid_days'):
        return True
    if col in {'d_events','d_sessions','d_pages_unique','d_active_hours','d_nextsong_cnt','d_listen_sec_sum','d_level_free_cnt','d_level_paid_cnt','d_song_bucket_unique','d_artist_bucket_unique'}:
        return True
    return False

count_like = [c for c in feature_cols if _is_count_like(str(c))]
print('count_like:', len(count_like))
clip_q = 0.995
clip_upper = train_ud_t[count_like].quantile(clip_q) if len(count_like) else None

def _log1p_clip(df: pd.DataFrame) -> None:
    if not count_like:
        return
    x = df[count_like]
    x = x.clip(lower=0)
    x = x.clip(upper=clip_upper, axis=1)
    df[count_like] = np.log1p(x)

_log1p_clip(train_ud_t)
_log1p_clip(val_ud_t)
_log1p_clip(test_ud_t)

scaler = StandardScaler()
train_ud_t[feature_cols] = scaler.fit_transform(train_ud_t[feature_cols])
val_ud_t[feature_cols] = scaler.transform(val_ud_t[feature_cols])
test_ud_t[feature_cols] = scaler.transform(test_ud_t[feature_cols])

train_ud_t[feature_cols] = train_ud_t[feature_cols].astype(np.float32)
val_ud_t[feature_cols] = val_ud_t[feature_cols].astype(np.float32)
test_ud_t[feature_cols] = test_ud_t[feature_cols].astype(np.float32)

(train_ud_t.shape, val_ud_t.shape, test_ud_t.shape, len(feature_cols))

drop constant cols: 5
count_like: 47


((4872087, 105), (1233573, 105), (116160, 105), 98)

## 5) Transformer: build sequences + train (multi-seed soft vote)

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

train_sequences = build_user_day_sequences(train_ud_t, feature_cols, t_config.max_seq_len)
val_sequences = build_user_day_sequences(val_ud_t, feature_cols, t_config.max_seq_len)
test_sequences = build_user_day_sequences(test_ud_t, feature_cols, t_config.max_seq_len)
print('seqs:', len(train_sequences), len(val_sequences), len(test_sequences))

val_prob_by_seed = []
test_prob_by_seed = []
best_states = []

for seed in T_SEEDS:
    set_seed(int(seed))
    gen = torch.Generator()
    gen.manual_seed(int(seed))

    train_loader, val_loader = make_dataloaders(
        train_sequences,
        val_sequences,
        labels_dict,
        t_config,
        train_generator=gen,
    )

    test_keys = list(test_sequences.keys())
    test_dataset = UserSequenceDataset(test_sequences, labels=None, user_ids=test_keys)
    test_loader = DataLoader(
        test_dataset,
        batch_size=t_config.batch_size,
        shuffle=False,
        num_workers=t_config.num_workers,
        collate_fn=collate_batch,
    )

    model = ChurnTransformerUserDay(num_numeric=len(feature_cols), config=t_config).to(device)
    _, _, model, best_epoch, best_state = train_model(
        model,
        train_loader,
        val_loader,
        device,
        epochs=int(t_config.epochs),
        lr=float(t_config.lr),
        weight_decay=float(t_config.weight_decay),
        pos_weight=float(T_POS_WEIGHT),
        use_cosine_decay=bool(t_config.use_cosine_decay),
        eta_min_factor=float(t_config.eta_min_factor),
        warmup_epochs=int(t_config.warmup_epochs),
        use_focal_loss=bool(t_config.use_focal_loss),
        focal_gamma=float(t_config.focal_gamma),
        early_stop_patience=int(t_config.early_stop_patience),
        early_stop_min_delta=float(t_config.early_stop_min_delta),
    )

    val_probs = predict_proba(model, val_loader, device)
    test_probs = predict_proba(model, test_loader, device)
    val_prob_by_seed.append(val_probs)
    test_prob_by_seed.append(test_probs)
    best_states.append({'seed': int(seed), 'best_epoch': int(best_epoch), 'state_dict': best_state})

val_keys = list(val_sequences.keys())
test_keys = list(test_sequences.keys())
t_val_proba = np.mean([[d[k] for k in val_keys] for d in val_prob_by_seed], axis=0)
t_test_proba = np.mean([[d[k] for k in test_keys] for d in test_prob_by_seed], axis=0)
y_val = np.array([labels_dict[k] for k in val_keys], dtype=int)
t_val_auc = float(roc_auc_score(y_val, t_val_proba))
t_val_auc

build_user_day_sequences: 100%|██████████| 2904/2904 [00:00<00:00, 7016.57it/s]


seqs: 168003 42537 2904


train_model:   0%|          | 0/25 [00:00<?, ?it/s]/Data/yuhan.wu/miniconda3/envs/py310/lib/python3.10/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
predict_proba: 100%|██████████| 12/12 [00:00<00:00, 270.98it/s]


0.7609384344556035

: 

## 6) XGB: expand user-day into fixed-length per-user features + train

In [ ]:
def _to_day(value: str | pd.Timestamp | np.datetime64) -> np.datetime64:
    if isinstance(value, np.datetime64):
        return value.astype('datetime64[D]')
    return np.datetime64(pd.Timestamp(value).normalize().to_datetime64(), 'D')

def build_window_features_from_df(
    user_day: pd.DataFrame,
    *,
    lookback_days: int,
    end_day: str | pd.Timestamp,
    fill_value: float = 0.0,
    dtype: str = 'float32',
) -> pd.DataFrame:
    lookback = int(lookback_days)
    user_day = user_day.sort_values(['userId', 'day'], kind='mergesort')
    end_d = _to_day(end_day)
    start_d = end_d - np.timedelta64(lookback - 1, 'D')
    day_d = pd.to_datetime(user_day['day']).values.astype('datetime64[D]')
    in_window = (day_d >= start_d) & (day_d <= end_d)
    user_day = user_day.loc[in_window].copy()
    lag = (end_d - day_d[in_window]).astype(int)
    user_day['lag'] = lag.astype(np.int16)
    feature_cols_x = [c for c in user_day.columns if c not in {'userId', 'day', 'lag'}]
    wide = user_day.pivot(index='userId', columns='lag', values=feature_cols_x)
    full_cols = pd.MultiIndex.from_product([feature_cols_x, range(lookback)], names=['feature', 'lag'])
    wide = wide.reindex(columns=full_cols)
    pad = len(str(lookback - 1))
    wide.columns = [f"{feat}_t-{int(l):0{pad}d}" for feat, l in wide.columns]
    wide = wide.fillna(float(fill_value)).sort_index()
    if dtype:
        wide = wide.astype(dtype, copy=False)
    return wide

# 生成 XGB 用的 X_train/X_val
t0 = time.time()
all_X_train = []
all_y_train = []
all_X_val = []
all_y_val = []

for cutoff in tqdm(CUT_OFFS, desc='flatten cutoffs for xgb'):
    cutoff_str = str(pd.Timestamp(cutoff).date())
    paths = build_datasets_cli(
        train_path=str(TRAIN_PATH),
        test_path=str(TEST_PATH),
        output_dir=None,
        output_base_dir=str(OUTPUT_BASE_DIR),
        cache_dir=str(CACHE_DIR),
        cutoff_time=cutoff_str,
        horizon_days=int(HORIZON_DAYS),
        label_mode=str(LABEL_MODE),
        drop_last_k_events=int(DROP_LAST_K_EVENTS),
        exclude_pages=str(EXCLUDE_PAGES),
        exclude_auth_values=str(EXCLUDE_AUTH_VALUES),
        val_prob_low=float(VAL_PROB_LOW),
        val_prob_high=float(VAL_PROB_HIGH),
        split_seed=str(SPLIT_SEED),
        batch_size=int(BATCH_SIZE),
        full_calendar=bool(FULL_CALENDAR),
        force_recompute=bool(FORCE_RECOMPUTE),
        verbose=False,
    )
    train_ud = pd.read_parquet(paths['train_features'])
    val_ud = pd.read_parquet(paths['val_features'])
    train_lab = pd.read_parquet(paths['train_labels'])[['userId', 'label']]
    val_lab = pd.read_parquet(paths['val_labels'])[['userId', 'label']]

    end_day = pd.Timestamp(cutoff).normalize() - pd.Timedelta(days=1)
    start_day = end_day - pd.Timedelta(days=int(LOOKBACK_DAYS) - 1)
    def _slice(df: pd.DataFrame) -> pd.DataFrame:
        d = pd.to_datetime(df['day']).dt.normalize()
        return df.loc[(d >= start_day) & (d <= end_day)].copy()
    train_ud = _slice(train_ud)
    val_ud = _slice(val_ud)
    suffix = '|' + cutoff_str
    train_ud['userId'] = train_ud['userId'].astype(str) + suffix
    val_ud['userId'] = val_ud['userId'].astype(str) + suffix
    train_lab['userId'] = train_lab['userId'].astype(str) + suffix
    val_lab['userId'] = val_lab['userId'].astype(str) + suffix

    X_tr = build_window_features_from_df(train_ud, lookback_days=int(LOOKBACK_DAYS), end_day=end_day)
    X_va = build_window_features_from_df(val_ud, lookback_days=int(LOOKBACK_DAYS), end_day=end_day)
    y_tr = train_lab.set_index('userId')['label'].reindex(X_tr.index).astype('float')
    y_va = val_lab.set_index('userId')['label'].reindex(X_va.index).astype('float')
    keep_tr = y_tr.notna()
    keep_va = y_va.notna()
    X_tr, y_tr = X_tr.loc[keep_tr], y_tr.loc[keep_tr].astype(int)
    X_va, y_va = X_va.loc[keep_va], y_va.loc[keep_va].astype(int)
    all_X_train.append(X_tr)
    all_y_train.append(y_tr)
    all_X_val.append(X_va)
    all_y_val.append(y_va)

X_train = pd.concat(all_X_train, axis=0)
y_train = pd.concat(all_y_train, axis=0)
X_val = pd.concat(all_X_val, axis=0)
y_val_x = pd.concat(all_y_val, axis=0)
X_train, X_val = X_train.align(X_val, join='outer', axis=1, fill_value=0.0)
print('xgb train', X_train.shape, 'val', X_val.shape)
print('seconds', round(time.time() - t0, 2))

flatten cutoffs for xgb: 100%|██████████| 11/11 [00:25<00:00,  2.34s/it]


In [ ]:
train_params = dict(xgb_params)
train_params['scale_pos_weight'] = float(XGB_SCALE_POS_WEIGHT)
model = XGBClassifier(**train_params, callbacks=XGB_CALLBACKS)
model.fit(X_train, y_train, eval_set=[(X_val, y_val_x)], verbose=100)

x_val_proba = model.predict_proba(X_val)[:, 1]
x_val_auc = float(roc_auc_score(y_val_x, x_val_proba))
x_val_auc

## 7) XGB: train final model (train+val) + generate test probabilities

In [ ]:
best_iteration = getattr(model, 'best_iteration', None)
best_n_estimators = int(model.get_params()['n_estimators']) if best_iteration is None else int(best_iteration) + 1
X_all = pd.concat([X_train, X_val], axis=0)
y_all = pd.concat([y_train, y_val_x], axis=0)
final_params = dict(train_params)
final_params['n_estimators'] = best_n_estimators
final_model = XGBClassifier(**final_params)
final_model.fit(X_all, y_all, verbose=False)

X_test = build_window_features_from_df(test_ud, lookback_days=int(LOOKBACK_DAYS), end_day=end_day)
X_test = X_test.reindex(columns=X_all.columns, fill_value=0.0)
x_test_proba = final_model.predict_proba(X_test)[:, 1]
(best_n_estimators, X_test.shape, float(np.mean(x_test_proba)))

## 8) Ensembling: logit blend + maximize Balanced Accuracy on val

Approach:
- Align Transformer and XGB rolling val predictions (key: `userId|cutoff`)
- Grid search the blend weight `w`
- For each `w`, automatically pick the threshold `thr` that maximizes Balanced Accuracy
- Apply the best `w,thr` to test and generate the submission


In [ ]:
def _sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))

def _logit(p: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def best_balanced_accuracy_threshold(y: np.ndarray, p: np.ndarray) -> tuple[float, float]:
    y = np.asarray(y, dtype=np.int8)
    p = np.asarray(p, dtype=np.float64)
    pos = int(y.sum())
    neg = int(y.shape[0] - pos)
    if pos == 0 or neg == 0:
        return 0.5, 0.5
    order = np.argsort(p, kind='mergesort')
    p_sorted = p[order]
    y_sorted = y[order]
    n = y_sorted.shape[0]
    tp_suffix = np.cumsum(y_sorted[::-1], dtype=np.int64)[::-1]
    tp = np.concatenate([tp_suffix, np.array([0], dtype=np.int64)])
    pred_pos = (n - np.arange(n + 1)).astype(np.int64)
    fp = pred_pos - tp
    fn = pos - tp
    tn = neg - fp
    tpr = tp / pos
    tnr = tn / neg
    bal = 0.5 * (tpr + tnr)
    best_i = int(np.argmax(bal))
    best_bal = float(bal[best_i])
    if best_i <= 0:
        best_thr = 0.0
    elif best_i >= n:
        best_thr = 1.0
    else:
        best_thr = float(p_sorted[best_i])
    return best_thr, best_bal

# 1) 对齐 rolling val
t_val_df = pd.DataFrame({'userId': np.array(val_keys, dtype=str), 'label': y_val.astype(int), 'proba_t': t_val_proba.astype(float)})
x_val_df = pd.DataFrame({'userId': X_val.index.astype(str), 'label': y_val_x.astype(int).to_numpy(), 'proba_x': x_val_proba.astype(float)})
val_join = t_val_df.merge(x_val_df, on=['userId', 'label'], how='inner')
print('val overlap', val_join.shape)

y = val_join['label'].to_numpy().astype(int)
z_t = _logit(val_join['proba_t'].to_numpy())
z_x = _logit(val_join['proba_x'].to_numpy())

def eval_w(w: float) -> tuple[float, float, float]:
    z = float(w) * z_t + (1.0 - float(w)) * z_x
    p = _sigmoid(z)
    thr, bal = best_balanced_accuracy_threshold(y, p)
    return float(w), float(thr), float(bal)

coarse_ws = np.linspace(0.0, 1.0, int(BLEND_W_GRID_COARSE))
coarse = [eval_w(w) for w in coarse_ws]
w0, thr0, bal0 = max(coarse, key=lambda t: t[2])

lo = max(0.0, w0 - float(BLEND_W_FINE_SPAN))
hi = min(1.0, w0 + float(BLEND_W_FINE_SPAN))
fine_ws = np.linspace(lo, hi, int(BLEND_W_GRID_FINE))
fine = [eval_w(w) for w in fine_ws]
best_w, best_thr, best_bal = max(fine, key=lambda t: t[2])

best_w, best_thr, best_bal

## 9) Generate submission (id,target) + report individual/blended Balanced Accuracy (val)

In [ ]:
# Baseline: each model's best threshold (for comparison only)
t_thr, t_bal = best_balanced_accuracy_threshold(y_val, t_val_proba)
x_thr, x_bal = best_balanced_accuracy_threshold(y_val_x.to_numpy(), x_val_proba)
print('transformer val bal:', t_bal, 'thr:', t_thr)
print('xgb         val bal:', x_bal, 'thr:', x_thr)
print('blend       val bal:', best_bal, 'thr:', best_thr, 'w:', best_w)

# test 融合（按 id 对齐）
t_test_df = pd.DataFrame({'id': np.array(test_keys, dtype=int), 'proba_t': t_test_proba.astype(float)})
x_test_df = pd.DataFrame({'id': X_test.index.astype(int), 'proba_x': x_test_proba.astype(float)})
test_join = t_test_df.merge(x_test_df, on='id', how='inner')
z = float(best_w) * _logit(test_join['proba_t'].to_numpy()) + (1.0 - float(best_w)) * _logit(test_join['proba_x'].to_numpy())
p_blend = _sigmoid(z)
pred = pd.DataFrame({'id': test_join['id'].to_numpy().astype(int), 'proba': p_blend.astype(float)})
example = pd.read_csv(EXAMPLE_SUB)
out = example[['id']].merge(pred, on='id', how='left')
out['proba'] = out['proba'].fillna(0.0)
out['target'] = (out['proba'] >= float(best_thr)).astype(int)
out = out[['id', 'target']]
print('test predicted positive rate:', float(out['target'].mean()))
out.head()

## 10) Save artifacts and submission files (experiments/ensemble/)

Keep the same output conventions as the other notebooks:
- Save config/meta and required files under `artifacts/run_name/`
- Save the final submission CSV under `submissions/`


In [ ]:
run_name = (
    f"ensemble_rolling_nb_cut{CUTOFF_START}-{CUTOFF_END}_{CUTOFF_FREQ}_"
    f"sub{SUBMISSION_CUTOFF}_h{HORIZON_DAYS}_lb{LOOKBACK_DAYS}_"
    f"w{best_w:.3f}_thr{best_thr:.3f}_bal{best_bal:.6f}"
)

ART_DIR = ROOT / 'experiments' / 'ensemble' / 'artifacts' / run_name
SUB_DIR = ROOT / 'experiments' / 'ensemble' / 'submissions'
ART_DIR.mkdir(parents=True, exist_ok=True)
SUB_DIR.mkdir(parents=True, exist_ok=True)

# 保存 transformer（每个 seed 的 best_state）
for item in best_states:
    sd = {k: v for k, v in item.items() if k != 'state_dict'}
    torch.save(item['state_dict'], ART_DIR / f"transformer_model_seed{item['seed']}.pt")
    (ART_DIR / f"transformer_model_seed{item['seed']}.json").write_text(json.dumps(sd, ensure_ascii=False, indent=2), encoding='utf-8')
(ART_DIR / 'transformer_training_config.json').write_text(json.dumps(asdict(t_config), ensure_ascii=False, indent=2), encoding='utf-8')
(ART_DIR / 'transformer_feature_columns.json').write_text(json.dumps(list(feature_cols), ensure_ascii=False, indent=2), encoding='utf-8')
scaler_payload = {'mean_': scaler.mean_.tolist(), 'scale_': scaler.scale_.tolist(), 'var_': scaler.var_.tolist(), 'n_features_in_': int(getattr(scaler, 'n_features_in_', len(feature_cols)))}
(ART_DIR / 'transformer_scaler.json').write_text(json.dumps(scaler_payload, ensure_ascii=False, indent=2), encoding='utf-8')

# 保存 xgb 模型
final_model.save_model(ART_DIR / 'xgb_model.json')
(ART_DIR / 'xgb_feature_columns.json').write_text(json.dumps(list(X_all.columns), ensure_ascii=False, indent=2), encoding='utf-8')

# 保存融合与预测
val_join.to_parquet(ART_DIR / 'pred_val_join.parquet', index=False)
pd.DataFrame({'id': test_join['id'].to_numpy().astype(int), 'proba_t': test_join['proba_t'].to_numpy().astype(float), 'proba_x': test_join['proba_x'].to_numpy().astype(float), 'proba_blend': pred['proba'].to_numpy().astype(float)}).to_parquet(ART_DIR / 'pred_test_join.parquet', index=False)

meta = {
    'run_name': run_name,
    'cutoffs': [str(pd.Timestamp(c).date()) for c in CUT_OFFS],
    'submission_cutoff': SUBMISSION_CUTOFF,
    'lookback_days': int(LOOKBACK_DAYS),
    'horizon_days': int(HORIZON_DAYS),
    'label_mode': str(LABEL_MODE),
    'transformer': {'seeds': [int(s) for s in T_SEEDS], 'pos_weight': float(T_POS_WEIGHT), 'val_auc': float(t_val_auc), 'val_bal_acc': float(t_bal), 'thr': float(t_thr)},
    'xgb': {'scale_pos_weight': float(XGB_SCALE_POS_WEIGHT), 'val_auc': float(x_val_auc), 'val_bal_acc': float(x_bal), 'thr': float(x_thr), 'best_n_estimators': int(best_n_estimators), 'params': train_params},
    'blend': {'best_w': float(best_w), 'best_thr_bal': float(best_thr), 'best_bal_acc': float(best_bal), 'method': 'logit_blend_gridsearch'},
}
(ART_DIR / 'train_meta.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')

sub_path = SUB_DIR / f"{run_name}.csv"
out.to_csv(sub_path, index=False)

ART_DIR, sub_path

In [ ]:
# Optional: submit to Kaggle (requires ~/.kaggle/kaggle.json)
import sys
sys.path.append(f"{base_path}/src")
sys.path.append(base_path)
from kaggle_submit import submit_and_track

competition = 'churn-prediction-25-26'
message = 'logist ensemble rolling nb'
info = submit_and_track(
    sub_path,
    competition=competition,
    message=message,
)
info